In [1]:
# Install required packages
%pip install ultralytics roboflow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 138.9 MB/s eta 0:00:0000:01


In [2]:
# Verify GPU is available
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU device     : {torch.cuda.get_device_name(0) 
                          if torch.cuda.is_available() else 'None'}")

CUDA available : True
GPU device     : Tesla T4


In [4]:
# Download base PPE dataset

from roboflow import Roboflow


rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(27)
dataset = version.download("yolov8", location="data/raw/base_dataset")



print("Base dataset downloaded successfully.")      

In [ ]:
# Download coustom annotated dataset


from roboflow import Roboflow

rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("zukoos-workspace").project("construction-safety-custom")
version = project.version(3)
dataset = version.download("yolov8", location="data/raw/custom_dataset")

print("Custom dataset downloaded successfully.")




In [6]:
# IVerify class ordering in both downloaded datasets
# before running data preparation

import yaml

with open("data/raw/base_dataset/data.yaml", "r") as f:
    base_yaml = yaml.safe_load(f)

with open("data/raw/custom_dataset/data.yaml", "r") as f:
    custom_yaml = yaml.safe_load(f)

print("Base dataset classes:")
for i, name in enumerate(base_yaml["names"]):
    print(f"  {i}: {name}")

print("\nCustom dataset classes:")
for i, name in enumerate(custom_yaml["names"]):
    print(f"  {i}: {name}")

In [ ]:
# RUn the data preparation pipeline
!python src/data_preparation.py


# Verify the output structure
import os
for split in ["train", "val", "test"]:
    img_count = len(os.listdir(f"data/processed/images/{split}"))
    lbl_count = len(os.listdir(f"data/processed/labels/{split}"))
    print(f"{split:5s} — images: {img_count:4d} | labels: {lbl_count:4d}")



In [ ]:
# Confirm dataset.yaml is correct before training
with open("data/processed/dataset.yaml", "r") as f:
    content = yaml.safe_load(f)

print("dataset.yaml contents:")
for key, value in content.items():
    print(f"  {key}: {value}")

In [ ]:
# Launch YOLOv8 fine-tuning
!python src/train.py

In [ ]:
# Display training results
from IPython.display import Image, display
import glob

results_dir = "outputs/training_runs/ppe_detection_v1"

# Show results plot (auto-generated by Ultralytics)
results_plot = f"{results_dir}/results.png"
display(Image(results_plot))

# Show confusion matrix
confusion_matrix = f"{results_dir}/confusion_matrix.png"
display(Image(confusion_matrix))

# Show F1 curve
f1_curve = f"{results_dir}/F1_curve.png"
display(Image(f1_curve))
